# MenuEye · OCR 벤치마크 (Colab GPU)

CPU에서 느린 v5 서버-검출을 **GPU로 빠르게** 돌려 v3 vs v5 복원율을 측정한다(정확도 그대로).

## 먼저 (중요)
1. 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 선택
2. 아래 셀을 위에서부터 순서대로 실행
3. 결과 `benchmark.csv` 등을 내려받아 로컬에 두면, 나머지(차트·문서)는 로컬에서 마무리

## 1. GPU 확인 (Tesla T4 나오면 OK)

In [ ]:
!nvidia-smi

## 2. 설치 (PaddlePaddle GPU + PaddleOCR)
2~3분 소요. ERROR로 멈추면 메시지 복사해 두기.
※ Colab CUDA에 따라 인덱스(cu126/cu123/cu118)가 다를 수 있음. 실패 시 아래 주석 참고.

In [ ]:
# GPU 빌드 (Colab CUDA 12.x). 실패하면 cu123 또는 cu118로 바꿔보세요.
!python -m pip install -q "paddlepaddle-gpu==3.1.0" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ \
  || python -m pip install -q "paddlepaddle-gpu==3.1.0" -i https://www.paddlepaddle.org.cn/packages/stable/cu123/
!pip install -q paddleocr
print("설치 완료. 혹시 numpy 관련 에러가 뜨면 [런타임 → 세션 다시 시작] 후 3번부터 이어서 실행.")

## 3. 설치 확인 (GPU=True 여야 빠름. False면 CPU로 돌아가 느림)

In [ ]:
import paddle, paddleocr
print("paddle", paddle.__version__, "| paddleocr", paddleocr.__version__)
print("GPU 사용 가능:", paddle.device.is_compiled_with_cuda())

## 4. 번들 업로드 & 압축 해제
로컬 `menueye_colab_bundle.zip`(스크립트+이미지25+truth25)을 업로드.

In [ ]:
from google.colab import files
up = files.upload()   # menueye_colab_bundle.zip 선택
import zipfile, os
zname = list(up.keys())[0]
with zipfile.ZipFile(zname) as z:
    z.extractall("/content/")
!ls /content/menueye_v2/data/menu_images/menu/한국 | head
!ls /content/menueye_v2/data/truth_*.txt | wc -l

## 5. 벤치마크 실행 (v3 vs v5, 25장) — GPU면 수 분
이미지별로 표 줄이 하나씩 찍힘. 순수 OCR이라 API 불필요.

In [ ]:
import os
os.environ["FLAGS_enable_pir_api"] = "0"
!cd /content/menueye_v2 && python scripts/benchmark_ocr.py

## 6. (EDA용) 인식 요약도 생성 — ocr_summary.csv / ocr_lines.csv

In [ ]:
!cd /content/menueye_v2 && python scripts/run_ocr.py --lang korean --version PP-OCRv5

## 7. 결과 내려받기 (로컬에 두면 차트·문서는 로컬에서 마무리)

In [ ]:
from google.colab import files
import shutil, os
res = "/content/menueye_v2/data/ocr_results"
out = "/content/benchmark_results.zip"
import zipfile
with zipfile.ZipFile(out, "w") as z:
    for f in ["benchmark.csv", "ocr_summary.csv", "ocr_lines.csv"]:
        p = os.path.join(res, f)
        if os.path.exists(p):
            z.write(p, f)
print("=== benchmark.csv ===")
print(open(os.path.join(res, "benchmark.csv"), encoding="utf-8-sig").read())
files.download(out)